# Sentiment analysis on Twiter


In [1]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer

# 1️⃣ Cargar CSV con codificación correcta
dat = pd.read_csv("sa.csv", header=None, sep=";", encoding="latin1")
texts = dat[0].astype(str).tolist()  # columna de textos
labels = dat[1].values           # columna de labels

# 2️⃣ Parámetros
maxlen = 25       # longitud máxima (no se usa directamente aquí)
max_words = 5000  # máximo de palabras a considerar

# 3️⃣ Tokenización
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(texts)
word_index = tokenizer.word_index
print(f"Found {len(word_index)} unique tokens.")

# 4️⃣ Codificar textos a matriz binaria (como en R: texts_to_matrix mode="binary")
encoding = tokenizer.texts_to_matrix(texts, mode='binary')  # forma (num_samples, max_words)
print(encoding.shape)  # ejemplo: (2000, 5000)

# 5️⃣ Reshape para añadir una dimensión extra como en R
encoding = np.expand_dims(encoding, axis=-1)  # forma (num_samples, max_words, 1)

# 6️⃣ Convertir labels a array de numpy
labels = np.array(labels)

# 7️⃣ Dividir en train y test
np.random.seed(42)
num_samples = encoding.shape[0]
training_samples = 1500

# Seleccionar índices de entrenamiento de forma aleatoria
training_indices = np.random.choice(num_samples, training_samples, replace=False)

x_train = encoding[training_indices, :, :]
y_train = labels[training_indices]

x_test = np.delete(encoding, training_indices, axis=0)
y_test = np.delete(labels, training_indices, axis=0)

print(x_train.shape, y_train.shape)
print(x_test.shape, y_test.shape)


2026-03-18 23:38:27.248899: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773873507.260368  498291 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773873507.264243  498291 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-18 23:38:27.276623: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Found 3258 unique tokens.
(2000, 5000)
(1500, 5000, 1) (1500,)
(500, 5000, 1) (500,)


# Modelo 1

In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
# Modelo 1
# Definir modelo secuencial 1D
model1 = models.Sequential([
    layers.Input(shape=(5000, 1)),
    # Capa Conv1D
    layers.Conv1D(
        filters=128,
        kernel_size=50,
        activation='relu',
        padding='same'
    ),
    # Max Pooling
    layers.MaxPooling1D(pool_size=25),

    # Aplanar
    layers.Flatten(),

    # Capa densa intermedia
    layers.Dense(25, activation='relu'),
    layers.Dropout(0.4),

    # Capa de salida
    layers.Dense(1, activation='sigmoid')
])

# Mostrar resumen del modelo
model1.summary()


I0000 00:00:1773873509.180531  498291 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22314 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:01:00.0, compute capability: 8.6


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 5000, 128)      │         6,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 200, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25600)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 25)             │       640,025 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 25)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            26 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 646,579 (2.47 MB)

 Trainable params: 646,579 (2.47 MB)

 Non-trainable params: 0 (0.00 B)

In [3]:
model1.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])

In [4]:
model1.fit(x_train,y_train,epochs=25,batch_size=32,validation_split=0.2)

Epoch 1/25


I0000 00:00:1773873510.353933  498356 service.cc:148] XLA service 0x7aa844008990 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773873510.353960  498356 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 3090, Compute Capability 8.6
2026-03-18 23:38:30.372051: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1773873510.444379  498356 cuda_dnn.cc:529] Loaded cuDNN version 90800


25/38 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4451 - loss: 0.6971 

I0000 00:00:1773873511.519211  498356 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 45ms/step - accuracy: 0.4557 - loss: 0.6962 - val_accuracy: 0.4800 - val_loss: 0.6947
Epoch 2/25
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5503 - loss: 0.6898 - val_accuracy: 0.5767 - val_loss: 0.6755
Epoch 3/25
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5988 - loss: 0.6654 - val_accuracy: 0.5900 - val_loss: 0.6645
Epoch 4/25
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6464 - loss: 0.6411 - val_accuracy: 0.6733 - val_loss: 0.6449
Epoch 5/25
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6286 - loss: 0.6230 - val_accuracy: 0.6633 - val_loss: 0.6334
Epoch 6/25
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6860 - loss: 0.5901 - val_accuracy: 0.6700 - val_loss: 0.6228
Epoch 7/25
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6872 - loss: 0.5875 - val_accuracy: 0.6333 - val_loss: 0.6319
Epoch 8/25
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7335 - loss: 0.5469 - val_accuracy: 0.6867 - val_loss: 0.6105
Ep

In [5]:
model1.evaluate(x_test,y_test)

16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6899 - loss: 0.7772


[0.7677037715911865, 0.6959999799728394]

# Modelo 2

In [6]:

# Definir modelo secuencial 1D
model2 = models.Sequential([
    # Capa densa inicial para reducir dimensionalidad
    layers.Input(shape=(5000,)),
    layers.Dense(100),

    # Reshape para Conv1D
    layers.Reshape((100, 1)),

    # Conv1D
    layers.Conv1D(
        filters=128,
        kernel_size=10,
        activation='relu',
        padding='same'
    ),

    # Max Pooling
    layers.MaxPooling1D(pool_size=25),

    # Aplanar
    layers.Flatten(),

    # Capa densa intermedia
    layers.Dense(25, activation='relu'),
    layers.Dropout(0.4),

    # Capa de salida para clasificación binaria
    layers.Dense(1, activation='sigmoid')
])

# Mostrar resumen del modelo
model2.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 100)            │       500,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 100, 1)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 100, 128)       │         1,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 4, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 25)             │        12,825 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 25)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            26 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 514,359 (1.96 MB)

 Trainable params: 514,359 (1.96 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
model2.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])
model2.fit(x_train,y_train,epochs=25,batch_size=32,validation_split=0.2)
model2.evaluate(x_test,y_test)

Epoch 1/25


2026-03-18 23:38:42.252007: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_768', 8 bytes spill stores, 8 bytes spill loads



38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 39ms/step - accuracy: 0.4980 - loss: 0.6936 - val_accuracy: 0.4700 - val_loss: 0.6930
Epoch 2/25
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5761 - loss: 0.6805 - val_accuracy: 0.7200 - val_loss: 0.6420
Epoch 3/25
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8235 - loss: 0.5519 - val_accuracy: 0.8167 - val_loss: 0.4754
Epoch 4/25
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9100 - loss: 0.3161 - val_accuracy: 0.8067 - val_loss: 0.3991
Epoch 5/25
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9712 - loss: 0.1430 - val_accuracy: 0.8233 - val_loss: 0.4272
Epoch 6/25
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9886 - loss: 0.0561 - val_accuracy: 0.8167 - val_loss: 0.5069
Epoch 7/25
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9988 - loss: 0.0232 - val_accuracy: 0.8167 - val_loss: 0.5367
Epoch 8/25
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9991 - loss: 0.0128 - val_accuracy: 0.8367 - val_loss: 0.6110
Ep

[1.3518013954162598, 0.8399999737739563]